In [ ]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

orders = pd.read_csv(
    "data/olist_delivery_clean.csv",
    parse_dates=["order_purchase_timestamp"]
)

print(orders.shape)
print(orders.columns.tolist())

In [ ]:
features = ["same_state", "estimated_delivery_days", "product_weight_g"]

X = orders[features]
y = orders["is_late"]

print(X.dtypes)
print(X.isna().sum())

print(y.value_counts())

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

K = 5

model = make_pipeline(
    SimpleImputer(strategy="median"),  
    StandardScaler(),                   
    KNeighborsClassifier(n_neighbors=K) 
)

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score

def evaluate_splits(X, y, splits, model):
    rows = []

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        fold_model = clone(model)
        fold_model.fit(X_train, y_train)
        predictions = fold_model.predict(X_valid)
        probabilities = fold_model.predict_proba(X_valid)[:, 1]
        rows.append({
            "fold": fold,
            "roc_auc": roc_auc_score(y_valid, probabilities),
            "precision": precision_score(y_valid, predictions, zero_division=0),
            "recall": recall_score(y_valid, predictions, zero_division=0),
        })

    return pd.DataFrame(rows)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

random_results = evaluate_splits(X, y, kf.split(X), model)
random_results

In [ ]:
metrics = ["roc_auc", "precision", "recall"]

random_results[metrics].agg(["mean", "std"])

In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

group_results = evaluate_splits(
    X,
    y,
    gkf.split(X, y, groups=orders["seller_id"]),
    model
)

group_results

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

orders_time = orders.sort_values(
    "order_purchase_timestamp",
    kind="stable"
).reset_index(drop=True)
X_time = orders_time[features]
y_time = orders_time["is_late"]

tscv = TimeSeriesSplit(n_splits=5)

time_results = evaluate_splits(
    X_time,
    y_time,
    tscv.split(X_time),
    model
)
time_results

In [ ]:
def summarize(metric):
    table = pd.DataFrame(
        {
            "KFold": random_results[metric].to_numpy(),
            "GroupKFold": group_results[metric].to_numpy(),
            "TimeSeriesSplit": time_results[metric].to_numpy(),
        },
        index=["fold_1", "fold_2", "fold_3", "fold_4", "fold_5"]
    ).T
    mean = table.mean(axis=1)
    std = table.std(axis=1)

    table["mean"] = mean
    table["std"] = std

    return table.round(4)


for metric in ["roc_auc", "precision", "recall"]:
    print(metric)
    display(summarize(metric))

В работе использовал KNN с числом соседей `k = 5` и тремя признаками из M0: `same_state`, `estimated_delivery_days` и `product_weight_g`

В весе товара было 16 пропусков. Я заполнил их медианой, потому что на неё меньше влияют очень большие значения, чем на среднее. Также применил StandardScaler: вес измеряется в граммах, срок — в днях, и без масштабирования величина чисел сильно влияет на расстояния в KNN. Медиана и параметры масштабирования вычислялись только на обучающей части каждого фолда. К проверочной части применялись готовые значения, чтобы её данные не влияли на обучение.

Я сравнил KFold, GroupKFold и TimeSeriesSplit, в каждой схеме использовал пять фолдов. Для GroupKFold выбрал группировку по продавцу, чтобы проверить модель на заказах продавцов, которых не было в обучении. Для TimeSeriesSplit отсортировал заказы по дате покупки: обучение происходило на прошлом, а проверка — на следующем участке данных. Обучающая часть постепенно расширялась. Пять проверок позволили оценить результат на нескольких последовательных участках истории.

Средний ROC-AUC составил 0.5456 для KFold, 0.5423 для GroupKFold и 0.5360 для TimeSeriesSplit. Стандартное отклонение между фолдами — 0.0080, 0.0061 и 0.0264.
Для основной оценки и отчёта компании я бы выбрал TimeSeriesSplit, потому что она наиболее естественно воспроизводит будущее использование модели: обучение на прошлых заказах и предсказание следующих

Пока модель получилась слабой... Во временной проверке средний recall равен примерно 0.96%, а precision — 10.81%. Получается, модель пропускает почти все реальные опоздания, а большинство её предупреждений неверные Я предполагаю, что одна из причин - признаки, на которых она обучалась

При случайном KFold модель может обучаться на заказах, которые появились
позже проверяемых. В реальной работе результаты будущих заказов
на момент прогноза ещё неизвестны, поэтому оценка может быть завышена.

Кроме того, при KFold заказы одного продавца могут попасть и в обучение,
и в проверку. GroupKFold разделяет продавцов между этими частями, но не
сохраняет порядок времени.

Средний ROC-AUC KFold и GroupKFold различается всего на 0.0033,
при стандартных отклонениях 0.0080 и 0.0061. По этим результатам
нельзя утверждать, что разница вызвана именно продавцами